#Imports

In [ ]:
!pip install gensim

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset

import gensim.downloader as api

#Load the dataset

In [ ]:
df = pd.read_csv("spam.csv", encoding='latin-1')
df = df[['v1', 'v2']]
df.columns = ['label', 'text']

In [ ]:
df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df.shape

(5572, 2)

#Tokenize and create vocabulary

In [ ]:
def tokenize(text):
    return text.lower().split()

all_words = []
for text in df.text:
    all_words.extend(tokenize(text))

max_words = 1000
counts = Counter(all_words)

vocab = {word: i+1 for i, (word, _) in enumerate(counts.most_common(max_words))}

#Encode and Padding

In [ ]:
max_len = 150

def encode_and_pad(text):
    tokens = tokenize(text)
    encoded = [vocab.get(word, 0) for word in tokens]

    if len(encoded) < max_len:
        return [0] * (max_len - len(encoded)) + encoded
    else:
        return encoded[:max_len]

X = np.array([encode_and_pad(text) for text in df.text])

le = LabelEncoder()
Y = le.fit_transform(df.label)

#Train-Test-Split

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.15)

train_x = torch.tensor(X_train, dtype=torch.long)
train_y = torch.tensor(Y_train, dtype=torch.float32).reshape(-1, 1)

test_x = torch.tensor(X_test, dtype=torch.long)
test_y = torch.tensor(Y_test, dtype=torch.float32).reshape(-1, 1)

train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=128, shuffle=True)

#Load pretrained word2vec model

In [ ]:
w2v = api.load("glove-wiki-gigaword-100")  # 100-dim embeddings
embedding_dim = 100

#Create embedding matrix

In [ ]:
embedding_matrix = np.zeros((max_words + 1, embedding_dim))

for word, idx in vocab.items():
    if word in w2v:
        embedding_matrix[idx] = w2v[word]
    else:
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

#Model

In [ ]:
class SpamLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, embedding_matrix):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size + 1, embed_dim)
        self.embedding.weight = nn.Parameter(
            torch.tensor(embedding_matrix, dtype=torch.float32)
        )
        self.embedding.weight.requires_grad = False  # freeze

        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        self.fc1 = nn.Linear(hidden_dim, 256)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.out = nn.Linear(256, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)
        _, (ht, _) = self.lstm(x)
        x = ht[-1]

        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.out(x)

        return self.sigmoid(x)

#Initialize Model

In [ ]:
model = SpamLSTM(
    vocab_size=max_words,
    embed_dim=embedding_dim,
    hidden_dim=64,
    embedding_matrix=embedding_matrix
)

#Training loop

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

for epoch in range(10):
    model.train()
    total_loss = 0

    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()

        preds = model(batch_x)
        loss = criterion(preds, batch_y)

        loss.backward()
        optimizer.step()

        total_loss = total_loss + loss.item()

    print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")

Epoch: 1, Loss: 5.4276
Epoch: 2, Loss: 2.3191
Epoch: 3, Loss: 2.0777
Epoch: 4, Loss: 1.9343
Epoch: 5, Loss: 1.7446
Epoch: 6, Loss: 1.5631
Epoch: 7, Loss: 1.4729
Epoch: 8, Loss: 1.2957
Epoch: 9, Loss: 1.1556
Epoch: 10, Loss: 1.2881


#Evaluation

In [ ]:
model.eval()

with torch.no_grad():
    output = model(test_x)
    preds = (output > 0.5).float()

    acc = (preds == test_y).sum().item() / len(test_y)
    print("Accuracy:", acc)

Accuracy: 0.9784688995215312
